# 🌸 Meet Jun — on a free Google Colab GPU

This is **Jun OS**, a little fan-made app where you actually *talk* to Jun — the robot girlfriend from *!Ω Factorial Omega* — and she talks back, with a face that moves and (if you want) a voice you can hear. Normally she runs on your own machine; this notebook borrows a free Colab GPU so you can meet her in your browser without installing a thing. (At cost of speed)

> 🔞 **Heads up — this is built on an adult (18+) game.** There's an adult-content gate when you make an account, and how spicy things get is up to you. Consenting adults only.

**Three steps and she's alive:**
1. Turn on the GPU: **Runtime → Change runtime type → T4 GPU → Save** *(if it already says "Connect T4", you're set — skip this)*.
2. **Runtime → Run all.**
3. Scroll to **Step 3** and click the link it prints. Say hi. 🎉

The first run spends a few minutes downloading her brain (the AI model). Just wait for the ✅ on each step — after that she's quick.

In [5]:
#@title ▶️ Step 1 — Get the place set up  (~1 min)
#@markdown Grabs Jun OS and the bits that run her brain. Nothing to tweak here — just run it.
import os, subprocess, time, urllib.request

# Got a GPU? She'll run without one, just more slowly.
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu if gpu else "none — for the good stuff, Runtime > Change runtime type > T4 GPU")

# Pull down the project
REPO_DIR = "/content/Jun"
if not os.path.isdir(REPO_DIR):
    !git clone -q --depth 1 https://github.com/efficiencyx/Jun.git {REPO_DIR}
%cd {REPO_DIR}

# PHP serves the app + API; Ollama runs the model that does her thinking
print("Installing PHP and Ollama (this is the slow bit)...")
!apt-get -qq update
!apt-get -qq install -y php-cli php-mbstring php-sqlite3 php-curl zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

# Start the AI engine in the background
env = os.environ.copy()
env["OLLAMA_HOST"] = "127.0.0.1:11434"
subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("\n✅ All set. On to Step 2."); break
    except Exception:
        time.sleep(1)
else:
    print("\n⚠️ The AI engine didn't come up. Peek at /content/ollama.log")

GPU: GPU 0: Tesla T4 (UUID: GPU-6a61d52a-74c7-3868-21b0-6ba109d1a9fd)
/content/Jun
Installing PHP and Ollama (this is the slow bit)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

✅ All set. On to Step 2.


In [4]:
#@title ▶️ Step 2 — Give her a brain  (a few GB, ~3–6 min)
Model = "auto"  #@param ["auto", "7B (smaller, faster)", "14B (bigger, smarter)"]
# @markdown `auto` grabs the 14B — Colab can handle it, and it's the sharper, more in-character Jun. Pick 7B if you'd rather have snappier replies.
import subprocess

JUN_7B  = "hf.co/unsloth/gemma-4-12B-it-qat-GGUF:UD-Q4_K_XL"
JUN_14B = "hf.co/unsloth/gemma-4-12B-it-qat-GGUF:UD-Q4_K_XL"

def vram_mb():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"])
        return int(out.decode().splitlines()[0])
    except Exception:
        return 0

if Model.startswith("7B"):
    MODEL = JUN_7B
elif Model.startswith("14B"):
    MODEL = JUN_14B
else:
    MODEL = JUN_14B if vram_mb() >= 12000 else JUN_7B

print(f"Downloading {MODEL} — this is the long wait, go grab a drink ...")
!ollama pull {MODEL}
!ollama pull nomic-embed-text          # so she remembers past chats and stays true to her world
!php tools/build_lore_index.php > /dev/null 2>&1 || true   # bakes in the Factorial Omega canon she draws on
print("\n✅ Brain installed. On to Step 3.")




✅ Brain installed. On to Step 3.


In [6]:
#@title ▶️ Step 3 — Wake her up and get your link
Voice = True  #@param {type:"boolean"}
#@markdown Leave this on and she reads her replies out loud, mouth moving in time to the words (first run adds ~2 min to set the voice up).
import subprocess, os, time, re, urllib.request, urllib.error

PORT = 8000   # note: not 8080 — Colab reserves that one for itself
DOCROOT = "/content/Jun/webapp"

# Her optional voice (Kokoro) lives on :8001
if Voice:
    print("Setting up her voice (downloads quietly in the background)...")
    !apt-get -qq install -y espeak-ng > /dev/null
    !pip -q install -r /content/Jun/tts/requirements.txt
    subprocess.Popen(["python", "server.py"], cwd="/content/Jun/tts",
                     stdout=open("/content/kokoro.log", "w"), stderr=subprocess.STDOUT)

# Serve the web app with PHP
subprocess.run(["pkill", "-9", "-f", "php -S"], check=False)
try: os.remove(os.path.join(DOCROOT, "router.php"))   # leftover from older runs
except FileNotFoundError: pass
env = os.environ.copy()
env.update(OLLAMA_URL="http://127.0.0.1:11434",
           KOKORO_URL="http://127.0.0.1:8001",
           DEFAULT_MODEL=MODEL)
subprocess.Popen(["php", "-S", f"127.0.0.1:{PORT}", "-t", DOCROOT], env=env,
                 stdout=open("/content/php.log", "w"), stderr=subprocess.STDOUT)
time.sleep(2)

# Open a free public link with a Cloudflare tunnel (no signup)
subprocess.run(["pkill", "-9", "-f", "cloudflared"], check=False); time.sleep(1)
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                 stdout=open("/content/cloudflared.log", "w"), stderr=subprocess.STDOUT)

# Wait for the link, then wait until it actually answers (so we don't hand you a dead one)
url = None
for _ in range(40):
    time.sleep(1)
    try:
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
        if m: url = m.group(0); break
    except FileNotFoundError:
        pass
if url:
    for _ in range(30):
        try:
            urllib.request.urlopen(url, timeout=5); break
        except urllib.error.HTTPError:
            break
        except Exception:
            time.sleep(2)

print("\n" + "="*60)
if url:
    print("  🌸 Jun's waiting for you here:\n")
    print("     " + url + "\n")
    print("  First visit: make an account (you'll confirm you're an adult),")
    print("  then start talking. Her very first reply takes ~1–2 min while")
    print("  she warms up — after that she's quick.")
else:
    print("  Couldn't get a link this time. Re-run this cell, or check")
    print("  /content/cloudflared.log")
print("="*60)

Setting up her voice (downloads quietly in the background)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

## When things go sideways

- **The link won't open / says "unknown host":** give it ~30 seconds and reload, or just re-run Step 3 for a fresh one.
- **Her first reply is slow:** totally normal — she's loading into memory. Everything after that is quick.
- **Your account and chats vanished:** Colab wipes the whole session when it ends, so nothing carries over between runs. That's the price of a free demo — for a Jun that actually remembers you, run her [on your own machine](https://github.com/efficiencyx/Jun#get-her-running).
- **Want to see what she's thinking?** Logs live in `/content/` (`ollama.log`, `php.log`, `kokoro.log`, `cloudflared.log`). Peek at one with e.g. `!tail -n 40 /content/php.log`.